In [1]:
import pandas as pd

# Load the datasets
train_path = 'D:\\LLM-Driven_AI-Studio\\MLAgent\\data\\benchmark/DSEval/datasets/02_cardiovascular_diseases/train.csv'
test_path = 'D:\\LLM-Driven_AI-Studio\\MLAgent\\data\\benchmark/DSEval/datasets/02_cardiovascular_diseases/test.csv'

# Read the datasets
train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

# Drop rows with missing values
train_df_clean = train_df.dropna().copy()
test_df_clean = test_df.dropna().copy()

# Remove duplicates
train_df_clean = train_df_clean.drop_duplicates()
test_df_clean = test_df_clean.drop_duplicates()

# Display the first few rows of the cleaned datasets
train_df_clean.head(), test_df_clean.head()


(  General_Health  ... FriedPotato_Consumption
 0           Good  ...                    24.0
 1           Good  ...                     8.0
 2      Very Good  ...                    16.0
 3      Excellent  ...                     8.0
 4      Very Good  ...                     2.0
 
 [5 rows x 19 columns],
   General_Health  ... FriedPotato_Consumption
 0      Very Good  ...                     4.0
 1           Good  ...                     2.0
 2      Very Good  ...                     4.0
 3      Very Good  ...                     8.0
 4           Fair  ...                    30.0
 
 [5 rows x 19 columns])

In [2]:
from metagpt.tools.libs.data_preprocess import get_column_info

# Get column information for the cleaned training dataset
column_info_train = get_column_info(train_df_clean)
print("Column information for the cleaned training dataset:")
print(column_info_train)

# Get column information for the cleaned test dataset
column_info_test = get_column_info(test_df_clean)
print("Column information for the cleaned test dataset:")
print(column_info_test)


2025-08-30 18:20:09.001 | INFO     | metagpt.const:get_metagpt_package_root:29 - Package root set to D:\LLM-Driven_AI-Studio\MLAgent\experiments\DataInterpreter


Column information for the cleaned training dataset:
{'Category': ['General_Health', 'Checkup', 'Exercise', 'Heart_Disease', 'Skin_Cancer', 'Other_Cancer', 'Depression', 'Diabetes', 'Arthritis', 'Sex', 'Age_Category', 'Smoking_History'], 'Numeric': ['Height_(cm)', 'Weight_(kg)', 'BMI', 'Alcohol_Consumption', 'Fruit_Consumption', 'Green_Vegetables_Consumption', 'FriedPotato_Consumption'], 'Datetime': [], 'Others': []}
Column information for the cleaned test dataset:
{'Category': ['General_Health', 'Checkup', 'Exercise', 'Heart_Disease', 'Skin_Cancer', 'Other_Cancer', 'Depression', 'Diabetes', 'Arthritis', 'Sex', 'Age_Category', 'Smoking_History'], 'Numeric': ['Height_(cm)', 'Weight_(kg)', 'BMI', 'Alcohol_Consumption', 'Fruit_Consumption', 'Green_Vegetables_Consumption', 'FriedPotato_Consumption'], 'Datetime': [], 'Others': []}


In [3]:
import pandas as pd

# Copy the cleaned datasets to avoid modifying the original ones
train_df_copy = train_df_clean.copy()
test_df_copy = test_df_clean.copy()

# Define a function to categorize BMI
def categorize_bmi(bmi):
    if bmi < 18.5:
        return 'Underweight'
    elif 18.5 <= bmi < 25:
        return 'Normal weight'
    elif 25 <= bmi < 30:
        return 'Overweight'
    else:
        return 'Obesity'

# Apply the function to create the new feature 'BMI_Category'
train_df_copy['BMI_Category'] = train_df_copy['BMI'].apply(categorize_bmi).astype('category')
test_df_copy['BMI_Category'] = test_df_copy['BMI'].apply(categorize_bmi).astype('category')

# Display the first few rows of the updated datasets
train_df_copy.head(), test_df_copy.head()


(  General_Health  ...   BMI_Category
 0           Good  ...        Obesity
 1           Good  ...  Normal weight
 2      Very Good  ...  Normal weight
 3      Excellent  ...     Overweight
 4      Very Good  ...  Normal weight
 
 [5 rows x 20 columns],
   General_Health  ...   BMI_Category
 0      Very Good  ...     Overweight
 1           Good  ...        Obesity
 2      Very Good  ...     Overweight
 3      Very Good  ...  Normal weight
 4           Fair  ...        Obesity
 
 [5 rows x 20 columns])

In [4]:
from metagpt.tools.libs.data_preprocess import get_column_info

# Using the latest DataFrame variables from the finished tasks
column_info_train = get_column_info(train_df_copy)
column_info_test = get_column_info(test_df_copy)

print("Train DataFrame Column Info:")
print(column_info_train)
print("\nTest DataFrame Column Info:")
print(column_info_test)


Train DataFrame Column Info:
{'Category': ['General_Health', 'Checkup', 'Exercise', 'Heart_Disease', 'Skin_Cancer', 'Other_Cancer', 'Depression', 'Diabetes', 'Arthritis', 'Sex', 'Age_Category', 'Smoking_History'], 'Numeric': ['Height_(cm)', 'Weight_(kg)', 'BMI', 'Alcohol_Consumption', 'Fruit_Consumption', 'Green_Vegetables_Consumption', 'FriedPotato_Consumption'], 'Datetime': [], 'Others': ['BMI_Category']}

Test DataFrame Column Info:
{'Category': ['General_Health', 'Checkup', 'Exercise', 'Heart_Disease', 'Skin_Cancer', 'Other_Cancer', 'Depression', 'Diabetes', 'Arthritis', 'Sex', 'Age_Category', 'Smoking_History'], 'Numeric': ['Height_(cm)', 'Weight_(kg)', 'BMI', 'Alcohol_Consumption', 'Fruit_Consumption', 'Green_Vegetables_Consumption', 'FriedPotato_Consumption'], 'Datetime': [], 'Others': ['BMI_Category']}


In [5]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

# Identify categorical columns
categorical_columns = ['General_Health', 'Checkup', 'Exercise', 'Sex', 'Age_Category', 'Smoking_History', 'BMI_Category']

# Label encode categorical columns
label_encoders = {}
for col in categorical_columns:
    le = LabelEncoder()
    train_df_copy[col] = le.fit_transform(train_df_copy[col])
    test_df_copy[col] = le.transform(test_df_copy[col])
    label_encoders[col] = le

# Split the training data into features and target
X_train = train_df_copy.drop(columns=['Heart_Disease'])
y_train = train_df_copy['Heart_Disease']

# Split the test data into features and target
X_test = test_df_copy.drop(columns=['Heart_Disease'])
y_test = test_df_copy['Heart_Disease']

# Initialize and train the XGBoost classifier
xgb_model = XGBClassifier(use_label_encoder=False, eval_metric='logloss', n_estimators=100, max_depth=5, learning_rate=0.1)
xgb_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred_proba = xgb_model.predict_proba(X_test)[:, 1]

# Compute the area under the ROC curve
roc_auc = roc_auc_score(y_test, y_pred_proba)
roc_auc


ValueError: Invalid classes inferred from unique values of `y`.  Expected: [0 1], got ['No' 'Yes']

In [6]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

# Identify categorical columns
categorical_columns = ['General_Health', 'Checkup', 'Exercise', 'Sex', 'Age_Category', 'Smoking_History', 'BMI_Category', 'Heart_Disease']

# Label encode categorical columns
label_encoders = {}
for col in categorical_columns:
    le = LabelEncoder()
    train_df_copy[col] = le.fit_transform(train_df_copy[col])
    test_df_copy[col] = le.transform(test_df_copy[col])
    label_encoders[col] = le

# Split the training data into features and target
X_train = train_df_copy.drop(columns=['Heart_Disease'])
y_train = train_df_copy['Heart_Disease']

# Split the test data into features and target
X_test = test_df_copy.drop(columns=['Heart_Disease'])
y_test = test_df_copy['Heart_Disease']

# Initialize and train the XGBoost classifier
xgb_model = XGBClassifier(use_label_encoder=False, eval_metric='logloss', n_estimators=100, max_depth=5, learning_rate=0.1)
xgb_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred_proba = xgb_model.predict_proba(X_test)[:, 1]

# Compute the area under the ROC curve
roc_auc = roc_auc_score(y_test, y_pred_proba)
roc_auc

ValueError: DataFrame.dtypes for data must be int, float, bool or category. When categorical type is supplied, the experimental DMatrix parameter`enable_categorical` must be set to `True`.  Invalid columns:Skin_Cancer: object, Other_Cancer: object, Depression: object, Diabetes: object, Arthritis: object

In [7]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

# Identify categorical columns
categorical_columns = ['General_Health', 'Checkup', 'Exercise', 'Sex', 'Age_Category', 'Smoking_History', 'BMI_Category', 'Heart_Disease', 'Skin_Cancer', 'Other_Cancer', 'Depression', 'Diabetes', 'Arthritis']

# Label encode categorical columns
label_encoders = {}
for col in categorical_columns:
    le = LabelEncoder()
    train_df_copy[col] = le.fit_transform(train_df_copy[col])
    test_df_copy[col] = le.transform(test_df_copy[col])
    label_encoders[col] = le

# Split the training data into features and target
X_train = train_df_copy.drop(columns=['Heart_Disease'])
y_train = train_df_copy['Heart_Disease']

# Split the test data into features and target
X_test = test_df_copy.drop(columns=['Heart_Disease'])
y_test = test_df_copy['Heart_Disease']

# Initialize and train the XGBoost classifier
xgb_model = XGBClassifier(use_label_encoder=False, eval_metric='logloss', n_estimators=100, max_depth=5, learning_rate=0.1)
xgb_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred_proba = xgb_model.predict_proba(X_test)[:, 1]

# Compute the area under the ROC curve
roc_auc = roc_auc_score(y_test, y_pred_proba)
roc_auc

D:\LLM-Driven_AI-Studio\MLAgent\experiments\DataInterpreter\venv\lib\site-packages\xgboost\core.py:158: UserWarning: [18:22:11] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-06abd128ca6c1688d-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


0.835509052160248